# 실험 2. 평점 요약 변수

- 기준: `exp01`의 기준 모델 (LightGBM 5-Fold, OOF AUC 0.95866)
- 0점은 "해당 없음"이므로 평균·최솟값은 0을 제외하고 계산
- 방법: 파생 변수를 추가하고 같은 Fold, 같은 파라미터로 OOF AUC 비교

## 환경설정

In [1]:
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
START_TIME = time.time()

## 데이터 경로 탐색

In [2]:
REQUIRED_FILES = ["train.csv", "test.csv", "sample_submission.csv"]


def has_all_files(folder: Path) -> bool:
    return all((folder / name).is_file() for name in REQUIRED_FILES)

# 크로스플랫폼 경로 자동 해결

def find_data_dir() -> Path:
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        candidates = sorted({p.parent for p in kaggle_root.glob("**/train.csv") if has_all_files(p.parent)})
    else:
        local = [p.resolve() for p in (Path("data"), Path("../data")) if has_all_files(p)]
        candidates = list(dict.fromkeys(local))  # drop duplicates, keep order

    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(
            f"데이터 폴더를 찾지 못했습니다. {REQUIRED_FILES} 세 파일이 모두 있는 폴더가 필요합니다.\n"
            "- Kaggle: 오른쪽 Input 패널에서 대회 데이터를 추가하세요.\n"
            "- 로컬: 노트북과 같은 위치의 data/ 폴더에 파일을 넣으세요."
        )
    raise RuntimeError(
        "데이터 폴더 후보가 여러 개입니다. 사용할 폴더를 DATA_DIR에 직접 지정하세요:\n"
        + "\n".join(f"  - {c}" for c in candidates)
    )


DATA_DIR = find_data_dir()
print("DATA_DIR =", DATA_DIR)

DATA_DIR = /Users/youngjin/ai/kaggle_practice/Kaggle_Airline_Satisfaction/data


## 데이터 로드 및 전처리

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

TARGET = "satisfaction"
CAT_COLS = ["Gender", "Customer Type", "Type of Travel", "Class"]
NUM_COLS = [c for c in train.columns if c not in CAT_COLS + ["id", TARGET]]
RATING_COLS = [c for c in NUM_COLS if train[c].max() <= 5]

In [4]:
y = train[TARGET].astype(int)
X = train.drop(columns=["id", TARGET]).copy()
X_test = test.drop(columns=["id"]).copy()

for col in CAT_COLS:
    categories = sorted(train[col].unique())
    X[col] = pd.Categorical(X[col], categories=categories)
    X_test[col] = pd.Categorical(X_test[col], categories=categories)

assert list(X.columns) == list(X_test.columns)
print("X      :", X.shape)
print("X_test :", X_test.shape)
print("y mean :", round(y.mean(), 4))

X      : (699635, 21)
X_test : (299844, 21)
y mean : 0.4436


## 교차검증 함수

In [5]:
LGB_PARAMS = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "seed": SEED,
    "verbose": -1,
}
N_FOLDS = 5


def run_cv(X, X_test, params=LGB_PARAMS):
    """Stratified 5-fold LightGBM; returns the OOF AUC and the averaged test prediction."""
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof_pred = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        dtrain = lgb.Dataset(X.iloc[tr_idx], label=y.iloc[tr_idx])
        dvalid = lgb.Dataset(X.iloc[va_idx], label=y.iloc[va_idx], reference=dtrain)

        model = lgb.train(
            params,
            dtrain,
            num_boost_round=5000,
            valid_sets=[dvalid],
            callbacks=[lgb.early_stopping(100, verbose=False)],
        )

        oof_pred[va_idx] = model.predict(X.iloc[va_idx], num_iteration=model.best_iteration)
        test_pred += model.predict(X_test, num_iteration=model.best_iteration) / N_FOLDS
        print(f"Fold {fold}: AUC = {roc_auc_score(y.iloc[va_idx], oof_pred[va_idx]):.5f}   trees = {model.best_iteration}")

    return roc_auc_score(y, oof_pred), test_pred

In [6]:
results = {}

## 기준 모델 (원본 변수)

In [7]:
auc, _ = run_cv(X, X_test)
results["기준 (원본 변수)"] = auc
print(f"\nOOF AUC = {auc:.5f}")

Fold 1: AUC = 0.95877   trees = 549


Fold 2: AUC = 0.95759   trees = 293


Fold 3: AUC = 0.95930   trees = 286


Fold 4: AUC = 0.95870   trees = 280


Fold 5: AUC = 0.95902   trees = 256

OOF AUC = 0.95866


## 파생 변수 생성

In [8]:
def rating_features(df):
    ratings = df[RATING_COLS].where(df[RATING_COLS] > 0)  # treat 0 (N/A) as missing
    return pd.DataFrame({
        "rating_mean": ratings.mean(axis=1),
        "rating_min": ratings.min(axis=1),
        "zero_count": (df[RATING_COLS] == 0).sum(axis=1),
    })


F = rating_features(X)
F_test = rating_features(X_test)
F.describe().round(2)

,rating_mean,rating_min,zero_count
count,699635.00,699635.00,699635.00
mean,3.33,1.73,0.10
std,0.67,0.83,0.47
min,1.00,1.00,0.00
25%,2.85,1.00,0.00
50%,3.38,2.00,0.00
75%,3.85,2.00,0.00
max,5.00,5.00,6.00


## 실험 2-1: rating_mean 추가

- 이유: 13개 평점을 하나로 요약. 0점(해당 없음)을 제외한 평균이 타깃과 상관 +0.53으로 단일 평점보다 높음

In [9]:
FEATS = ["rating_mean"]
auc, _ = run_cv(pd.concat([X, F[FEATS]], axis=1), pd.concat([X_test, F_test[FEATS]], axis=1))
results["rating_mean 추가"] = auc
print(f"\nOOF AUC = {auc:.5f}   (기준 대비 {auc - results['기준 (원본 변수)']:+.5f})")

Fold 1: AUC = 0.95870   trees = 346


Fold 2: AUC = 0.95741   trees = 291


Fold 3: AUC = 0.95928   trees = 291


Fold 4: AUC = 0.95853   trees = 208


Fold 5: AUC = 0.95892   trees = 311

OOF AUC = 0.95856   (기준 대비 -0.00010)


## 실험 2-2: rating_min 추가

- 이유: 한 항목이라도 나쁘면 불만일 것이라는 가설. 0 제외 최저 평점의 타깃 상관 +0.35

In [10]:
FEATS = ["rating_min"]
auc, _ = run_cv(pd.concat([X, F[FEATS]], axis=1), pd.concat([X_test, F_test[FEATS]], axis=1))
results["rating_min 추가"] = auc
print(f"\nOOF AUC = {auc:.5f}   (기준 대비 {auc - results['기준 (원본 변수)']:+.5f})")

Fold 1: AUC = 0.95873   trees = 360


Fold 2: AUC = 0.95752   trees = 399


Fold 3: AUC = 0.95934   trees = 284


Fold 4: AUC = 0.95870   trees = 437


Fold 5: AUC = 0.95897   trees = 288

OOF AUC = 0.95863   (기준 대비 -0.00003)


## 실험 2-3: zero_count 추가

- 이유: 0점이 3개 이상이면 만족 비율 0.86 이상으로 급등. 개별 열로는 잡기 어려운 패턴을 한 변수로 제공

In [11]:
FEATS = ["zero_count"]
auc, _ = run_cv(pd.concat([X, F[FEATS]], axis=1), pd.concat([X_test, F_test[FEATS]], axis=1))
results["zero_count 추가"] = auc
print(f"\nOOF AUC = {auc:.5f}   (기준 대비 {auc - results['기준 (원본 변수)']:+.5f})")

Fold 1: AUC = 0.95893   trees = 511


Fold 2: AUC = 0.95750   trees = 265


Fold 3: AUC = 0.95946   trees = 416


Fold 4: AUC = 0.95858   trees = 322


Fold 5: AUC = 0.95895   trees = 284

OOF AUC = 0.95868   (기준 대비 +0.00002)


## 실험 2-4: 세 변수 모두 추가

- 이유: 세 변수를 함께 넣었을 때 조합 효과가 있는지 확인

In [12]:
FEATS = ["rating_mean", "rating_min", "zero_count"]
auc, _ = run_cv(pd.concat([X, F[FEATS]], axis=1), pd.concat([X_test, F_test[FEATS]], axis=1))
results["전체 추가"] = auc
print(f"\nOOF AUC = {auc:.5f}   (기준 대비 {auc - results['기준 (원본 변수)']:+.5f})")

Fold 1: AUC = 0.95858   trees = 331


Fold 2: AUC = 0.95750   trees = 314


Fold 3: AUC = 0.95938   trees = 299


Fold 4: AUC = 0.95852   trees = 269


Fold 5: AUC = 0.95902   trees = 365

OOF AUC = 0.95859   (기준 대비 -0.00008)


## 결과 요약

In [13]:
summary = pd.Series(results, name="oof_auc").to_frame()
summary["diff"] = summary["oof_auc"] - results["기준 (원본 변수)"]
print(f"전체 실행 시간: {time.time() - START_TIME:.0f}초")
summary.round(5)

전체 실행 시간: 205초


,oof_auc,diff
기준 (원본 변수),0.95866,0.00000
rating_mean 추가,0.95856,-0.00010
rating_min 추가,0.95863,-0.00003
zero_count 추가,0.95868,0.00002
전체 추가,0.95859,-0.00008
